# Step 0: pyducklake primer

A short, self-contained tour of the DuckLake concepts and pyducklake functions used in the phase-1 tutorial. Everything runs on a tiny made-up `rides` table, so no case files are needed.

| Section | Concept | Functions |
|---|---|---|
| 1 | Catalog vs. data files | `Catalog`, `set_option`, `connection` |
| 2 | Namespaces, tables, schemas | `create_namespace`, `Schema.of`, `required`/`optional`, types, `create_table` |
| 3 | Writing | `append`, `begin_transaction`, `set_commit_message`, `delete` |
| 4 | Reading | `scan`, `select`, `with_limit`, `count`, `to_pandas`/`to_arrow`/`to_duckdb` |
| 5 | Snapshots | `snapshots`, `current_snapshot`, `inspect().snapshots()`, `inspect().files()` |
| 6 | Time travel | `with_snapshot`, `with_timestamp`, SQL `AT (VERSION => ...)` |
| 7 | Change capture | `table_insertions`, `table_changes` |
| 8 | SQL used in the tutorial | anti-join, `TRY_CAST`, `QUALIFY`, file hashes |
| 9 | Gotchas | |

**Install once:** `pip install "pyducklake[pandas]" pytz`

## 1. Catalog vs. data files

A DuckLake table lives in two places:

* the **catalog**, a SQL database holding the metadata: schemas, tables, columns, the list of data files, and snapshots. Here it is a DuckDB file, `metadata.ducklake`.
* the **data files**, Parquet files holding the rows, under `data/`.

`Catalog(name, uri, data_path=...)` connects to both. `name` becomes the SQL prefix: tables are called `primer.<schema>.<table>`.

> ⚠️ The folder of the catalog file must exist before you create the `Catalog`.

In [1]:
import hashlib
import shutil
from datetime import datetime
from decimal import Decimal
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
import pyarrow as pa
from pyducklake import (
    Catalog, Schema, required, optional,
    StringType, IntegerType, TimestampTZType, DecimalType, DateType,
)

BKK = ZoneInfo("Asia/Bangkok")
LAKE = Path.cwd() / "primer_lakehouse"

if LAKE.exists():                 # start clean on every run
    shutil.rmtree(LAKE)
LAKE.mkdir()

catalog = Catalog("primer", str(LAKE / "metadata.ducklake"), data_path=str(LAKE / "data"))

Two settings we always make:

* **Data inlining off.** By default DuckLake keeps small inserts *inside the catalog database* instead of writing a Parquet file. Turning it off makes every insert produce a file we can see.

Right now the lakehouse folder only holds the catalog. The `data/` folder appears with the first write.
* **Session time zone.** `catalog.connection` is the DuckDB connection underneath pyducklake. Setting its time zone makes timestamps display in Bangkok time.

In [2]:
catalog.set_option("data_inlining_row_limit", "0")

con = catalog.connection
con.execute("SET TimeZone = 'Asia/Bangkok'")

sorted(p.name for p in LAKE.iterdir())

['metadata.ducklake', 'metadata.ducklake.wal']

## 2. Namespaces, tables and schemas

A DuckLake **schema** is a group of tables; pyducklake calls it a **namespace**. In a medallion lakehouse, each layer is one namespace. Every catalog starts with `main`.

In [3]:
catalog.create_namespace("demo")
catalog.list_namespaces()

['demo', 'main']

A table schema is built with `Schema.of(...)`. Each column is either

* `required(name, type)`, meaning NOT NULL, or
* `optional(name, type)`, meaning NULL is allowed.

Common types: `StringType()`, `IntegerType()`, `DecimalType(precision, scale)`, `DateType()`, `TimestampTZType()` (timestamp with time zone).

In [4]:
rides_schema = Schema.of(
    required("ride_id", StringType()),
    required("zone", StringType()),
    required("requested_at", TimestampTZType()),
    optional("fare_thb", DecimalType(10, 2)),     # NULL for a cancelled ride
)

rides = catalog.create_table(("demo", "rides"), rides_schema)   # identifier = (namespace, table)
print(catalog.list_tables("demo"))
rides.schema

[('demo', 'rides')]


Schema(fields=(NestedField(field_id=1, name='ride_id', type=StringType(), required), NestedField(field_id=2, name='zone', type=StringType(), required), NestedField(field_id=3, name='requested_at', type=TimestampTZType(), required), NestedField(field_id=4, name='fare_thb', type=DecimalType(precision=10, scale=2), optional)), schema_id=0)

`load_table` returns a handle to an existing table, for example in a new session. A `Table` object is only a handle; the data stays in the lakehouse.

In [5]:
same_table = catalog.load_table(("demo", "rides"))
same_table.namespace, same_table.name

('demo', 'rides')

## 3. Writing

### 3.1 `append`

`append` takes a **PyArrow table**. Arrow types map onto DuckLake types: `pa.string()` → String, `pa.decimal128(10, 2)` → Decimal, `pa.timestamp("us", tz=...)` → TimestampTZ.

In [6]:
def ride_batch(ids, zones, hours, fares):
    # Build a PyArrow table with the same column order as demo.rides.
    return pa.table({
        "ride_id": pa.array(ids, pa.string()),
        "zone": pa.array(zones, pa.string()),
        "requested_at": pa.array([datetime(2026, 4, 1, h, 0, tzinfo=BKK) for h in hours], pa.timestamp("us", tz="Asia/Bangkok")),
        "fare_thb": pa.array([None if f is None else Decimal(f) for f in fares], pa.decimal128(10, 2)),
    })


batch1 = ride_batch(["R1", "R2", "R3"], ["Siam", "Asok", "Ari"], [8, 8, 18], ["120.00", None, "95.50"])
rides.append(batch1)
rides.scan().to_pandas()

,ride_id,zone,requested_at,fare_thb
0,R1,Siam,2026-04-01 08:00:00+07:00,120.0
1,R2,Asok,2026-04-01 08:00:00+07:00,NaN
2,R3,Ari,2026-04-01 18:00:00+07:00,95.5


> ⚠️ `append` matches columns **by position, not by name**. The Arrow table must list its columns in the same order as the table schema.

A `required` column rejects NULLs:

In [7]:
bad = ride_batch(["R9"], ["Siam"], [8], ["50.00"]).set_column(1, "zone", pa.array([None], pa.string()))
try:
    rides.append(bad)
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

ConstraintException -> Constraint Error: NOT NULL constraint failed: rides.zone


### 3.2 Transactions and commit messages

`catalog.begin_transaction()` groups writes into **one atomic commit**. It commits when the `with` block ends normally and rolls back if an exception is raised. Inside a transaction, `set_commit_message` attaches an author and a message to the commit.

In [8]:
with catalog.begin_transaction() as txn:
    catalog.set_commit_message("load evening rides", author="primer")
    txn.load_table(("demo", "rides")).append(ride_batch(["R4", "R5"], ["Silom", "Siam"], [18, 18], ["140.00", "88.00"]))

print("rows:", rides.scan().count())

rows: 5


If anything inside the block fails, nothing is written:

In [9]:
try:
    with catalog.begin_transaction() as txn:
        txn.load_table(("demo", "rides")).append(ride_batch(["R6"], ["Ari"], [8], ["70.00"]))
        raise RuntimeError("something went wrong after the append")
except RuntimeError as e:
    print("rolled back:", e)

print("rows:", rides.scan().count(), "(R6 was not kept)")

rolled back: something went wrong after the append
rows: 5 (R6 was not kept)


### 3.3 Other write operations

`delete(filter)`, `overwrite(df)` and `upsert(df, join_cols=[...])` also exist. The tutorial only appends; here is one delete, which we will meet again in Section 7:

In [10]:
rides.delete("ride_id = 'R2'")
rides.scan().to_pandas()

,ride_id,zone,requested_at,fare_thb
0,R1,Siam,2026-04-01 08:00:00+07:00,120.0
1,R3,Ari,2026-04-01 18:00:00+07:00,95.5
2,R4,Silom,2026-04-01 18:00:00+07:00,140.0
3,R5,Siam,2026-04-01 18:00:00+07:00,88.0


## 4. Reading

`table.scan()` creates a **scan**, a description of a query. Methods such as `select`, `with_limit` and `with_snapshot` each return a **new** scan, so you can chain them. Nothing is read until a terminal method runs:

| Terminal method | Returns |
|---|---|
| `count()` | number of rows |
| `to_pandas()` | pandas DataFrame |
| `to_arrow()` | PyArrow table (exact types) |
| `to_duckdb()` | DuckDB relation you can keep querying |

A filter is written as a SQL condition string.

In [11]:
evening = rides.scan("hour(requested_at) = 18").select("ride_id", "zone", "fare_thb")
print("evening rides:", evening.count())
evening.with_limit(2).to_pandas()

evening rides: 3


,ride_id,zone,fare_thb
0,R3,Ari,95.5
1,R4,Silom,140.0


pandas shows decimals as `float64`. Use Arrow to see the stored types:

In [12]:
print(rides.scan().to_pandas().dtypes, end="\n\n")
rides.scan().to_arrow().schema

ride_id                                  str
zone                                     str
requested_at    datetime64[us, Asia/Bangkok]
fare_thb                             float64
dtype: object



ride_id: string
zone: string
requested_at: timestamp[us, tz=Asia/Bangkok]
fare_thb: decimal128(10, 2)

In [13]:
rides.scan().to_duckdb().aggregate("zone, count(*) AS rides, sum(fare_thb) AS fare_thb", "zone").order("zone").df()

,zone,rides,fare_thb
0,Ari,1,95.5
1,Siam,2,208.0
2,Silom,1,140.0


Because DuckLake is a DuckDB extension, you can also query tables with plain SQL on the same connection:

In [14]:
con.sql("SELECT ride_id, fare_thb FROM primer.demo.rides WHERE fare_thb > 100").df()

,ride_id,fare_thb
0,R1,120.0
1,R4,140.0


## 5. Snapshots

Every committed change (create a namespace, create a table, append, delete) creates a **snapshot** with a new, increasing `snapshot_id`. Old Parquet files are never modified, so every snapshot can still be read.

> ⚠️ Snapshots belong to the **whole catalog**, not to one table. `rides.snapshots()` lists every snapshot in the catalog, including ones that did not touch `rides`.

In [15]:
rides.inspect().snapshots().to_pandas()[["snapshot_id", "snapshot_time", "changes", "author", "commit_message"]]

,snapshot_id,snapshot_time,changes,author,commit_message
0,0,2026-10-15 21:46:35.476552+07:00,"[(schemas_created, [main])]",NaN,NaN
1,1,2026-10-15 21:46:35.560186+07:00,"[(schemas_created, [demo])]",NaN,NaN
2,2,2026-10-15 21:46:35.605322+07:00,"[(tables_created, [demo.rides])]",NaN,NaN
3,3,2026-10-15 21:46:35.749487+07:00,"[(tables_inserted_into, [2])]",NaN,NaN
4,4,2026-10-15 21:46:35.828390+07:00,"[(tables_inserted_into, [2])]",primer,load evening rides
5,5,2026-10-15 21:46:35.921015+07:00,"[(tables_deleted_from, [2])]",NaN,NaN


Things to notice:

* the failed append (NULL zone) and the rolled-back transaction created **no** snapshot;
* `changes` refers to tables by **ID**, for example `tables_inserted_into: [2]`;
* the commit message appears only on the snapshot written inside a transaction.

Writing to a *second* table moves the catalog forward too, so `current_snapshot()` means "the latest snapshot in the catalog":

In [16]:
zones = catalog.create_table(("demo", "zones"), Schema.of(required("zone", StringType())))
zones.append(pa.table({"zone": ["Siam", "Asok", "Ari", "Silom"]}))

print("latest catalog snapshot:", rides.current_snapshot().snapshot_id, "(created by the zones table, not rides)")

latest catalog snapshot: 7 (created by the zones table, not rides)


To turn the table IDs in `changes` into names, use `ducklake_table_info()`:

In [17]:
con.sql("SELECT table_name, table_id, file_count, delete_file_count FROM ducklake_table_info('primer')").df()

,table_name,table_id,file_count,delete_file_count
0,rides,2,2,1
1,zones,3,1,0


`inspect().files()` lists the Parquet files of one table. Every append wrote one file, and the delete wrote a small *delete file* that marks R2 as removed without rewriting the data file.

In [18]:
files = rides.inspect().files().to_pandas()
for f in files.itertuples():
    print(Path(f.data_file).name, "| delete file:", Path(f.delete_file).name if isinstance(f.delete_file, str) else None)

ducklake-01a14007-7fed-78d7-ab71-3a1560ef31ab.parquet | delete file: ducklake-01a14007-809d-718c-ba04-e3898343e323-delete.parquet
ducklake-01a14007-8043-7b1b-b954-576aea3ed77e.parquet | delete file: None


## 6. Time travel

Let's record the current state, change the table, and then read the table as it was before the change.

In [19]:
before_snapshot = rides.current_snapshot().snapshot_id
before_time = datetime.now(BKK)

rides.append(ride_batch(["R7", "R8"], ["Asok", "Ari"], [8, 18], ["110.00", "99.00"]))

print("now                          :", rides.scan().count(), "rows")
print(f"with_snapshot({before_snapshot})              :", rides.scan().with_snapshot(before_snapshot).count(), "rows")
print("with_timestamp(before_time)  :", rides.scan().with_timestamp(before_time).count(), "rows")

now                          : 6 rows
with_snapshot(7)              : 4 rows
with_timestamp(before_time)  : 4 rows


The SQL form is `AT (VERSION => id)` or `AT (TIMESTAMP => ...)`. Going back to the very first insert also shows R2 again, from before the delete:

In [20]:
first_insert = con.sql(
    "SELECT min(snapshot_id) FROM primer.snapshots() WHERE map_contains(changes, 'tables_inserted_into')"
).fetchone()[0]
print("first insert snapshot:", first_insert)
con.sql(f"SELECT * FROM primer.demo.rides AT (VERSION => {first_insert}) ORDER BY ride_id").df()

first insert snapshot: 3


,ride_id,zone,requested_at,fare_thb
0,R1,Siam,2026-04-01 08:00:00+07:00,120.0
1,R2,Asok,2026-04-01 08:00:00+07:00,NaN
2,R3,Ari,2026-04-01 18:00:00+07:00,95.5


## 7. Change capture: what changed between snapshots?

* `table_insertions(start_snapshot, end_snapshot)` returns only the rows inserted in that range.
* `table_changes(...)` returns inserts **and** deletes (and updates), with a `change_type` column.

Both return a `ChangeSet`; call `.to_pandas()` or `.to_arrow()` on it.

In [21]:
rides.table_insertions(start_snapshot=before_snapshot + 1).to_pandas()

,ride_id,zone,requested_at,fare_thb
0,R7,Asok,2026-04-01 08:00:00+07:00,110.00
1,R8,Ari,2026-04-01 18:00:00+07:00,99.00


In [22]:
rides.table_changes(start_snapshot=first_insert).to_pandas().sort_values(["snapshot_id", "ride_id"])[["snapshot_id", "change_type", "ride_id", "zone", "fare_thb"]]

,snapshot_id,change_type,ride_id,zone,fare_thb
2,3,insert,R1,Siam,120.00
3,3,insert,R2,Asok,None
4,3,insert,R3,Ari,95.50
0,4,insert,R4,Silom,140.00
1,4,insert,R5,Siam,88.00
7,5,delete,R2,Asok,None
5,8,insert,R7,Asok,110.00
6,8,insert,R8,Ari,99.00


## 8. SQL patterns used in the tutorial

The tutorial writes its transformations in DuckDB SQL. Here are the four patterns on toy data.

**Anti-join with `NOT EXISTS`:** keep incoming rows whose key is not in the table yet. This is how bronze and silver avoid inserting the same trip twice.

In [23]:
incoming = ride_batch(["R7", "R8", "R10"], ["Asok", "Ari", "Siam"], [8, 18, 8], ["110.00", "99.00", "60.00"])
con.register("incoming", incoming)
con.sql('''
    SELECT * FROM incoming AS i
    WHERE NOT EXISTS (SELECT 1 FROM primer.demo.rides AS r WHERE r.ride_id = i.ride_id)
''').df()          # only R10 is new

,ride_id,zone,requested_at,fare_thb
0,R10,Siam,2026-04-01 08:00:00+07:00,60.0


**`TRY_CAST`** returns NULL instead of failing on a bad value, and **`timezone()`** converts to local Bangkok time before taking the date or hour.

In [24]:
con.sql('''
    SELECT raw,
           TRY_CAST(raw AS TIMESTAMPTZ)                                       AS parsed,
           CAST(timezone('Asia/Bangkok', TRY_CAST(raw AS TIMESTAMPTZ)) AS DATE) AS bangkok_date
    FROM (VALUES ('2026-04-01T23:30:00+07:00'), ('2026-04-01T23:30:00+00:00'), ('not a time')) AS t(raw)
''').df()

,raw,parsed,bangkok_date
0,2026-04-01T23:30:00+07:00,2026-04-01 23:30:00+07:00,2026-04-01
1,2026-04-01T23:30:00+00:00,2026-04-02 06:30:00+07:00,2026-04-02
2,not a time,NaT,NaT


**`QUALIFY`** filters on a window function. Here it keeps one row per key:

In [25]:
con.sql('''
    SELECT * FROM (VALUES ('R1', 1), ('R1', 2), ('R2', 1)) AS t(ride_id, version)
    QUALIFY row_number() OVER (PARTITION BY ride_id ORDER BY version DESC) = 1
    ORDER BY ride_id
''').df()

,ride_id,version
0,R1,2
1,R2,1


**File hash.** A SHA-256 hash of a file's bytes acts as a fingerprint: the same file always gives the same hash, and any change gives a different one. The tutorial stores it in bronze to detect a file that is loaded twice.

In [26]:
print(hashlib.sha256(b"ride_id\nR1\n").hexdigest()[:16])
print(hashlib.sha256(b"ride_id\nR1\n").hexdigest()[:16], "(same bytes -> same hash)")
print(hashlib.sha256(b"ride_id\nR2\n").hexdigest()[:16], "(different bytes -> different hash)")

a42ee51035d56540
a42ee51035d56540 (same bytes -> same hash)
c1f8286217c35f19 (different bytes -> different hash)


## 9. Gotchas to remember

| Gotcha | What to do |
|---|---|
| The catalog folder must exist before `Catalog(...)` | `mkdir` it first |
| `pytz` missing makes snapshot calls fail | `pip install pytz` |
| `append` matches columns by position | build Arrow tables in schema order |
| Snapshots are catalog-wide | `current_snapshot()` is the latest in the catalog, not the table's last change |
| `changes` shows table IDs | look IDs up with `ducklake_table_info('<catalog>')` |
| `with_timestamp()` drops the time zone of the datetime | set the session `TimeZone` and pass times in that zone |
| pandas shows `DECIMAL` as float and `DATE` as datetime | check `to_arrow().schema` |
| The catalog remembers `data_path` as an absolute path | reopen with the same path, or recreate the lakehouse |

You are ready for the phase-1 tutorial. 🚕